In [1]:
import pandas as pd  # Para manipulación de datos
import matplotlib.pyplot as plt  # Para visualización
import seaborn as sns  # Para gráficos más estilizados
import numpy as np
import os

In [2]:
# Cargar los datos
csv_file_path = r"../../data/nuevos/cleaned_data_nuevos.csv"
df = pd.read_csv(csv_file_path)

df.head()

,timestamp,open_binance,high_binance,low_binance,close_binance,volumen_binance,trades_binance,var_diaria_binance,rango_dia_binance,volatilidad_50_binance,...,momentum_200_dxy,SMA_50_dxy,EMA_50_dxy,SMA_200_dxy,EMA_200_dxy,RSI_dxy,pct_cambio_7d,tendencia_futura_7d,pct_cambio_30d,tendencia_futura_30d
0,2023-01-01,16541.77,16628.00,16499.01,16616.75,96925.41374,3218355.0,0.449451,128.99,367.529453,...,-1.084069,105.275233,105.547776,107.887225,106.544019,41.998608,NaN,0,NaN,0
1,2023-01-02,16617.17,16799.23,16548.70,16672.87,121888.57191,4036118.0,0.337732,250.53,362.064797,...,0.617581,105.229899,105.497667,107.890425,106.521392,45.834443,NaN,0,NaN,0
2,2023-01-03,16672.78,16778.40,16605.28,16675.18,159541.53733,5097596.0,0.013855,173.12,361.616039,...,-0.171920,105.187099,105.459327,107.889525,106.501478,49.435670,NaN,0,NaN,0
3,2023-01-04,16675.65,16991.87,16652.66,16850.36,220362.18862,6310703.0,1.050543,339.21,361.374263,...,-0.367944,105.144099,105.411903,107.887600,106.479075,45.887346,NaN,0,NaN,0
4,2023-01-05,16850.36,16879.82,16753.00,16831.85,163473.56641,4842014.0,-0.109849,126.82,360.953339,...,0.449461,105.119299,105.397318,107.889950,106.464756,55.868588,NaN,0,NaN,0


# Agrupar variables

In [3]:
sufijos = [
    "_binance",
    "_fg",
    "_gtrends",
    "_hashrate",
    "_mDbinance",
    "_sp500",
    "_nasdaq",
    "_dxy",
]

# Crear un diccionario para almacenar las columnas separadas
columnas_por_sufijo = {sufijo: [] for sufijo in sufijos}

# Recorrer las columnas y clasificarlas
for col in df.columns:
    for sufijo in sufijos:
        if col.endswith(sufijo):
            columnas_por_sufijo[sufijo].append(col)
            break

# Ahora tienes las columnas organizadas por sufijo
suma = 5 # timestamp, 4 columnas respuesta
for sufijo, columnas in columnas_por_sufijo.items():
    print(f"\nSufijo {sufijo} ({len(columnas)}):")
    print(columnas)
    suma += len(columnas)

assert suma == df.shape[1], f"Suma: {suma}, columnas:{df.shape[1]}"


Sufijo _binance (17):
['open_binance', 'high_binance', 'low_binance', 'close_binance', 'volumen_binance', 'trades_binance', 'var_diaria_binance', 'rango_dia_binance', 'volatilidad_50_binance', 'momentum_50_binance', 'volatilidad_200_binance', 'momentum_200_binance', 'SMA_50_binance', 'EMA_50_binance', 'SMA_200_binance', 'EMA_200_binance', 'RSI_binance']

Sufijo _fg (1):
['fear_greed_index_fg']

Sufijo _gtrends (1):
['bitcoin_gtrends']

Sufijo _hashrate (5):
['hashrate_hashrate', 'SMA_50_hashrate', 'SMA_200_hashrate', 'EMA_50_hashrate', 'EMA_200_hashrate']

Sufijo _mDbinance (17):
['open_mDbinance', 'high_mDbinance', 'low_mDbinance', 'close_mDbinance', 'volumen_mDbinance', 'trades_mDbinance', 'var_diaria_mDbinance', 'rango_dia_mDbinance', 'volatilidad_50_mDbinance', 'momentum_50_mDbinance', 'volatilidad_200_mDbinance', 'momentum_200_mDbinance', 'SMA_50_mDbinance', 'EMA_50_mDbinance', 'SMA_200_mDbinance', 'EMA_200_mDbinance', 'RSI_mDbinance']

Sufijo _sp500 (16):
['close_sp500', 'high_s

# Eliminación de variables redundantes o con información duplicada


## 1. Binance – precios OHLC
**Eliminar:**
- open_binance
- high_binance
- low_binance
- volumen_binance: se calcula close_binance * volume_binance

**Dejar:**
- close_binance
- volume_binance
- trades_binance
- rango_dia_binance: diferencia de high y low
- var_diaria_binance: cambio porcentual entre un valor y el valor inmediatamente anterior (en porcentaje)

In [4]:
df.drop(columns=['open_binance', 'high_binance', 'low_binance', 'volumen_binance'], inplace=True)

## 2. Binance - Medias móviles
Mantener una combinación de corto y largo plazo, evitando duplicar SMA y EMA para ambos.

**Eliminar:**
- SMA_50_binance
- EMA_200_binance

**Dejar:**
- EMA_50_binance
- SMA_200_binance

In [5]:
df.drop(columns=['SMA_50_binance', 'EMA_200_binance'], inplace=True)

## 3. Binance - Otras varibles
Conservaremos las variables de volatilidad, momentum y RSI

## 4. Indicadores
Conservaremos los indicadores: `fear_greed_index_fg` y `bitcoin_gtrends`

## 5. Hashrate
Conservaremos la variable `hashrate_hashrate` y mantendremos una combinación de corto y largo plazo para las medias móviles como en Binance

**Eliminar:**
- SMA_50_hashrate
- EMA_200_hashrate

**Dejar:**
- EMA_50_hashrate
- SMA_200_hashrate

In [6]:
df.drop(columns=['SMA_50_hashrate', 'EMA_200_hashrate'], inplace=True)

## 6. Mercado Derivados Binance, S&P 500, Nasdaq, DXY
Al compartir las mismas variables que Binance, repetiremos el proceso:

**Eliminar:**
- open
- high
- low
- SMA_50
- EMA_200

**Dejar:**
- close
- volume
- trades
- var_diaria
- rango dia
- volatilidad (50/200)
- momentum (50/200)
- SMA_200
- EMA_50

In [7]:
# mDbinance
df.drop(columns=[
    'open_mDbinance',
    'high_mDbinance',
    'low_mDbinance',
    'SMA_50_mDbinance',
    'EMA_200_mDbinance'
        ],
        inplace=True)

# S&P 500, Nasdaq, DXY
sufijos = ['_sp500', '_nasdaq', '_dxy']
columnas = ['open', 'high', 'low', 'SMA_50', 'EMA_200']
columnas_a_eliminar = [col + suf for col in columnas for suf in sufijos]

df.drop(columns=columnas_a_eliminar, inplace=True)

## 7. Varibles respuesta categóricas
Para mostrar la creación de la variable respuesta gráficamente se creó una variable categórica auxiliar que indicaba la tendencia alcista o bajista del bitcoin. Como no se va a realizar problemas de clasificación se eliminarán estas variables

In [8]:
df.drop(columns=['tendencia_futura_7d', 'tendencia_futura_30d'], inplace=True)

## 8. Conjunto de datos para 7 días
No se eliminará ninguna otra variable salvo la variable respuesta para 30 días

In [9]:
df_7d = df.drop(columns=['pct_cambio_30d'])

output_path = r"../../data/nuevos/dataset_7d_nuevos.csv"
os.makedirs(os.path.dirname(output_path), exist_ok=True)
df_7d.to_csv(output_path, index=False)

## 9. Conjunto de datos para 30 días
No se eliminará ninguna otra variable salvo la variable respuesta para 7 días

In [10]:
df_30d = df.drop(columns=['pct_cambio_7d'])

output_path = r"../../data/nuevos/dataset_30d_nuevos.csv"
os.makedirs(os.path.dirname(output_path), exist_ok=True)
df_30d.to_csv(output_path, index=False)

In [11]:
df.head()

,timestamp,close_binance,trades_binance,var_diaria_binance,rango_dia_binance,volatilidad_50_binance,momentum_50_binance,volatilidad_200_binance,momentum_200_binance,EMA_50_binance,...,rango_dia_dxy,volatilidad_50_dxy,momentum_50_dxy,volatilidad_200_dxy,momentum_200_dxy,EMA_50_dxy,SMA_200_dxy,RSI_dxy,pct_cambio_7d,pct_cambio_30d
0,2023-01-01,16616.75,3218355.0,0.449451,128.99,367.529453,-1.161843,2224.182129,-26.421555,17183.808952,...,1.049999,1.150009,-2.249096,2.899194,-1.084069,105.547776,107.887225,41.998608,NaN,NaN
1,2023-01-02,16672.87,4036118.0,0.337732,250.53,362.064797,2.100570,2233.911279,-18.275493,17163.772130,...,1.219999,1.143928,-2.127599,2.894822,0.617581,105.497667,107.890425,45.834443,NaN,NaN
2,2023-01-03,16675.18,5097596.0,0.013855,173.12,361.616039,0.335270,2243.310862,-18.533711,17144.611654,...,1.389999,1.129269,-2.006382,2.895847,-0.171920,105.459327,107.889525,49.435670,NaN,NaN
3,2023-01-04,16850.36,6310703.0,1.050543,339.21,361.374263,-0.297091,2251.609782,-11.177342,17133.072374,...,0.769997,1.123059,-2.020678,2.898148,-0.367944,105.411903,107.887600,45.887346,NaN,NaN
4,2023-01-05,16831.85,4842014.0,-0.109849,126.82,360.953339,1.014778,2259.475170,-18.188733,17121.259732,...,1.279999,1.111090,-1.166727,2.895634,0.449461,105.397318,107.889950,55.868588,NaN,NaN
